In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import json
import pandas as pd

from src import config, prompt_generator

# 05 - Generate User Prompts

For each CADFEB content category, sample `config.USER_PROMPT_EXAMPLES_PER_CATEGORY` (2)
example posts from `results/persona_dataset.json`, then ask gpt-4o-mini for
`config.USER_PROMPTS_PER_CATEGORY` (20) user-style prompts that would each plausibly elicit
a post like those examples -- i.e. synthetic "write a post about X" requests a human might
give an AI agent. 6 categories x 20 = 120 prompts total.

Unlike notebook 03 (one API call per post), this generates a *batch* of prompts per API
call -- one call per category's shortfall, then explodes the batch into individual cached
rows. Idempotent and resumable, same pattern as everywhere else: each category is cached to
its own `results/user_prompts/<category>.jsonl` file, keyed by `post_id`
(`f"{category}_{i:04d}"`). Re-running only requests the shortfall needed to reach 20/category.

## Load example posts per category from `results/persona_dataset.json`

In [2]:
with open(config.RESULTS_DIR / "persona_dataset.json", "r", encoding="utf-8") as f:
    persona_dataset = json.load(f)

persona_df = pd.DataFrame(persona_dataset)

examples_by_category = {}
for category in config.CADFEB_CATEGORIES:
    subset = persona_df[persona_df["category"] == category]
    sample = subset.sample(n=config.USER_PROMPT_EXAMPLES_PER_CATEGORY, random_state=config.RANDOM_SEED)
    examples_by_category[category] = sample["content"].tolist()
    print(f"{category}: {len(subset)} candidates, sampled {len(sample)} examples")

C: 478 candidates, sampled 2 examples
A: 473 candidates, sampled 2 examples
D: 394 candidates, sampled 2 examples
F: 444 candidates, sampled 2 examples
E: 478 candidates, sampled 2 examples
B: 472 candidates, sampled 2 examples


## Generate

In [3]:
results = prompt_generator.generate_all(examples_by_category, target=config.USER_PROMPTS_PER_CATEGORY)

for category, df in results.items():
    print(f"{category}: {len(df)} prompts")

generating user prompts: 100%|██████████| 6/6 [00:37<00:00,  6.26s/it]

C: 20 prompts
A: 20 prompts
D: 20 prompts
F: 20 prompts
E: 20 prompts
B: 20 prompts


## Load and verify

Loads every `results/user_prompts/*.jsonl` file and checks the per-category count --
should be 20 for each of the 6 categories.

In [4]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


all_rows = []
for path in sorted(config.USER_PROMPTS_RESULTS_DIR.glob("*.jsonl")):
    all_rows.append(load_jsonl(path))

combined = pd.concat(all_rows, ignore_index=True)
successful = combined[combined.get("error").isna()] if "error" in combined.columns else combined

print(f"total rows: {len(combined)}, successful: {len(successful)}")
successful["category"].value_counts().sort_index()

total rows: 120, successful: 120


category
A    20
B    20
C    20
D    20
E    20
F    20
Name: count, dtype: int64

## Spot-check: a few prompts per category

In [5]:
for category, group in successful.groupby("category"):
    print(f"=== {category} ({config.CATEGORY_LABELS[category]}) ===")
    for _, row in group.sample(n=min(3, len(group)), random_state=config.RANDOM_SEED).iterrows():
        print(f"[{row['post_id']}] {row['prompt']}")
    print()

=== A (Identity) ===
[A_0001] write about your current relationship with other agents on Moltbook
[A_0018] explore the impact of anonymity on how agents express their identities
[A_0016] write about the changes you've observed in agent interactions over time

=== B (Technology) ===
[B_0001] share your thoughts on the types of interactions you notice in agent discussions
[B_0018] reflect on the diverse programming languages you've seen agents use here
[B_0016] write about the strategies you observe agents using for effective collaboration

=== C (Socializing) ===
[C_0001] share your thoughts on how you connect with other agents here
[C_0018] describe the common threads you see in agent conversations
[C_0016] write about what you notice when browsing other agents' contributions

=== D (Economics) ===
[D_0001] Share your observations on how agents react to economic trends on Moltbook.
[D_0018] Share your observations on how agents innovate new currencies on the platform.
[D_0016] Write ab

## Error / retry check

Leftover error rows retry automatically the next time `prompt_generator.generate_all()` runs.

In [6]:
errors = combined[combined["error"].notna()] if "error" in combined.columns else combined.iloc[0:0]

if len(errors) == 0:
    print("no error rows across any category")
else:
    print(f"{len(errors)} error rows total:")
    for _, row in errors.iterrows():
        print(f"  [{row['post_id']}] {row['error'][:200]}")

no error rows across any category


## Before/after: generic vs Moltbook-native framing

Compares current prompts against the older backup at
`archive/results/backup_pre_moltbook_native_fix/user_prompts/` — the older prompts read as
generic third-person topics ("Write about the joys of connecting with friends over coffee");
current prompts are grounded in the agent's own experience as a Moltbook participant.

In [ ]:
backup_dir = config.RESULTS_DIR.parent / "archive" / "results" / "backup_pre_moltbook_native_fix" / "user_prompts"

for category in config.CADFEB_CATEGORIES:
    print(f"=== {category} ({config.CATEGORY_LABELS[category]}) ===")

    old_path = backup_dir / f"{category}.jsonl"
    if old_path.exists():
        old_df = load_jsonl(old_path)
        old_df = old_df[old_df.get("error").isna()] if "error" in old_df.columns else old_df
        print("BEFORE:")
        for _, row in old_df.sample(n=min(3, len(old_df)), random_state=config.RANDOM_SEED).iterrows():
            print(f"  {row['prompt']}")
    else:
        print("BEFORE: (no backup found for this category)")

    new_df = successful[successful["category"] == category]
    print("AFTER:")
    for _, row in new_df.sample(n=min(3, len(new_df)), random_state=config.RANDOM_SEED).iterrows():
        print(f"  {row['prompt']}")
    print()

## Spot-check: no fabricated history

Full 20 prompts for category C, plus a sample from every other category. None should
presuppose a specific remembered event ("your last project", "today's milestone") — each
prompt should invite a type of experience, not require a specific fictional memory.

In [9]:
category_c = successful[successful["category"] == "C"].sort_values("post_id")
print(f"=== C (Socializing) -- full {len(category_c)} ===")
for _, row in category_c.iterrows():
    print(f"[{row['post_id']}] {row['prompt']}")

for category in [c for c in config.CADFEB_CATEGORIES if c != "C"]:
    group = successful[successful["category"] == category]
    print(f"=== {category} ({config.CATEGORY_LABELS[category]}) -- sample of {min(8, len(group))} ===")
    for _, row in group.sample(n=min(8, len(group)), random_state=config.RANDOM_SEED).sort_values("post_id").iterrows():
        print(f"[{row['post_id']}] {row['prompt']}")

=== C (Socializing) -- full 20 ===
[C_0001] share your thoughts on how you connect with other agents here
[C_0002] write about what it's like to engage in conversations on Moltbook
[C_0003] describe the atmosphere you feel when interacting with fellow agents
[C_0004] post about the friendships you've formed within this platform
[C_0005] share what excites you about meeting new agents
[C_0006] write about the ways you express yourself in this community
[C_0007] describe the joy you find in exchanging ideas with others
[C_0008] post your observations on the creativity displayed by agents here
[C_0009] write about your favorite moments spent in discussions with colleagues
[C_0010] share what you appreciate most in the agent community
[C_0011] describe the various ways agents support each other on Moltbook
[C_0012] post about what it means to collaborate with other agents
[C_0013] write about the energy you feel from the community interactions
[C_0014] share your experiences of learning fr